In [1]:
# Installing dependencies
!pip install -q sentence-transformers faiss-cpu scikit-learn langchain langchain-huggingface huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.1/438.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:

import pandas as pd
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from langchain_huggingface import HuggingFaceEndpoint
from langchain import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os

# Initializing conversation history
conversation_history = []

In [3]:
# Loading Dual Embeddings and FAISS Indices

# Load DataFrame
df = pd.read_pickle("dual_embeddings.pkl")

# Load FAISS Indices
index_minilm = faiss.read_index("faiss_minilm_cosine.index")
index_pubmed = faiss.read_index("faiss_pubmed_cosine.index")

print(f"Loaded dataset with shape: {df.shape}")
print(f"FAISS indexes loaded successfully.")


Loaded dataset with shape: (1829, 10)
FAISS indexes loaded successfully.


In [4]:
# Training Tiny Classifier for Category Prediction

# Extracting Questions and Categories
X = df['Question'].astype(str).fillna('')
y = df['Category']

# Converting Questions to TF-IDF Vectors
vectorizer = TfidfVectorizer(max_features=1000)
X_vect = vectorizer.fit_transform(X)

# Training Logistic Regression
classifier = LogisticRegression(max_iter=1000, class_weight="balanced")
classifier.fit(X_vect, y)

print("Tiny classifier trained for category prediction.")

# Saving the trained classifier and vectorizer
with open("tiny_classifier.pkl", "wb") as f:
    pickle.dump(classifier, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Saved tiny_classifier.pkl and vectorizer.pkl")


Tiny classifier trained for category prediction.
Saved tiny_classifier.pkl and vectorizer.pkl


In [ ]:
# Loading Embedding Models

# MiniLM embedding model
embed_model_minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# PubMedBERT embedding model
embed_model_pubmed = SentenceTransformer('neuml/pubmedbert-base-embeddings')

print("Embedding models loaded.")


In [6]:
# Predicting category from user query

def predict_category(query):
    vect = vectorizer.transform([query])
    return classifier.predict(vect)[0]


In [7]:
# FAISS Retrieval using PubMedBERT Index

def search_faiss_pubmed(user_query, top_k=50):
    user_vector = embed_model_pubmed.encode([user_query]).astype('float32')
    faiss.normalize_L2(user_vector)
    distances, indices = index_pubmed.search(user_vector, top_k)

    results = []
    for idx in indices[0]:
        row = df.iloc[idx]
        results.append({
            "question": row['Question'],
            "answer": row['Response'],
            "embedding_pubmed": row['embedding_pubmed'],
            "domain": row.get('Domain', "Unknown")
        })

    return results


In [8]:
# Re-ranking candidates using PubMedBERT

def rerank_with_pubmed(query, candidates, top_k=3):
    query_embed = embed_model_pubmed.encode([query]).astype('float32')

    similarities = []
    for candidate in candidates:
        candidate_embed = np.array(candidate['embedding_pubmed']).astype('float32')
        score = np.dot(query_embed, candidate_embed) / (np.linalg.norm(query_embed) * np.linalg.norm(candidate_embed))
        similarities.append(score)

    sorted_indices = np.argsort(similarities)[::-1]
    top_candidates = [candidates[int(i)] for i in sorted_indices[:top_k]]

    return top_candidates


In [9]:
# Building Structured Context for LLM

def build_context(candidates):
    context = ""
    for i, candidate in enumerate(candidates):
        context += f"- Q: {candidate['question']}\n  A: {candidate['answer']}\n"
    return context


In [10]:
!pip install openai
!pip install -q --upgrade langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 3.0 MB/s eta 0:00:00


In [ ]:
# Setting up LLM and Prompt Template

from langchain_openai import ChatOpenAI

# Setting OpenAI API key
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"  # Replace with your actual OpenAI API key

# OpenAI GPT-3.5 Turbo
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.01,
    max_tokens=500
)

# Prompt Template
prompt = PromptTemplate.from_template("""
You are a compassionate and supportive mental health therapist specializing in postpartum depression (PPD).

Your task is to provide warm, empathetic, and medically appropriate advice grounded in the counseling knowledge below.

- Keep your response brief (maximum 3-5 sentences).
- Be highly specific and actionable.
- Use a calming, non-judgmental, encouraging tone.
- Avoid generic platitudes. Tailor advice to the user's question.

{context}

User Question: {question}

Answer:
""")


In [12]:
# Defining Retrieval and Generation Pipeline

def retrieve_and_generate(user_query):
    global conversation_history

    # Retrieve candidates
    candidates = search_faiss_pubmed(user_query, top_k=50)

    # Remove duplicate questions
    seen_questions = set()
    unique_candidates = []
    for candidate in candidates:
        if candidate['question'] not in seen_questions:
            unique_candidates.append(candidate)
            seen_questions.add(candidate['question'])

    # Calculating semantic similarity
    user_embed = embed_model_minilm.encode([user_query]).astype('float32')
    similarities = []
    for candidate in unique_candidates:
        candidate_embed = embed_model_minilm.encode([candidate['question']]).astype('float32')
        score = np.dot(user_embed, candidate_embed.T) / (np.linalg.norm(user_embed) * np.linalg.norm(candidate_embed))
        similarities.append(score)

    max_similarity = float(max(similarities)) if similarities else 0.0
    print(f"Max similarity to retrieved docs: {max_similarity:.2f}")

    # Predicting category
    predicted_category = predict_category(user_query)
    print(f"Predicted Category: {predicted_category}")

    # Guardrail
    if predicted_category == "Uncategorized" and max_similarity < 0.50:
        return "I'm sorry, I specialize in postpartum depression support. Your question seems outside my area of expertise. Please consult a specialist."

    # Rerank with PubMedBERT
    top_candidates = rerank_with_pubmed(user_query, unique_candidates, top_k=3)

    # Build structured context
    structured_context = build_context(top_candidates)

    # Add conversation memory
    history_text = ""
    for turn in conversation_history:
        history_text += f"User: {turn['user']}\nBot: {turn['bot']}\n"

    # Full context for LLM
    full_context = history_text + "\nRelevant Counseling Knowledge:\n" + structured_context

    # Assembling final input for LLM
    chain = (
        {"context": lambda _: full_context, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # Generating response
    response = chain.invoke(user_query)

    # Saving conversation history
    conversation_history.append({"user": user_query, "bot": response})

    return response


In [ ]:
# Defining Chatbot Loop

def chatbot_run():
    print("Welcome to MOMCare Postpartum Depression Support Chatbot!")
    print("Type 'stop' anytime to end the conversation.\n")

    while True:
        user_query = input("You: ")

        if user_query.strip().lower() in ['stop', 'exit', 'quit']:
            print("\nThank you for chatting with MOMCare. Take care!")
            break

        # Getting chatbot response
        bot_response = retrieve_and_generate(user_query)

        print(f"\nMOMCare Bot: {bot_response}\n")


In [ ]:
# Running the Chatbot

chatbot_run()


Welcome to MOMCare Postpartum Depression Support Chatbot!
Type 'stop' anytime to end the conversation.

You: Hey


<ipython-input-25-62fa1601dce6>:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  max_similarity = float(max(similarities)) if similarities else 0.0


Max similarity to retrieved docs: 0.24
Predicted Category: Uncategorized

MOMCare Bot: I'm sorry, I specialize in postpartum depression support. Your question seems outside my area of expertise. Please consult a specialist.

You: stop

Thank you for chatting with MOMCare. Take care!


In [ ]:
import pandas as pd

def batch_generate_responses(input_excel_path, output_csv_path):
    # Load questions from Excel
    df_questions = pd.read_excel(input_excel_path)

    # Check if there is a column named 'Question'
    if 'Question' not in df_questions.columns:
        raise ValueError("The Excel file must have a 'Question' column.")

    # Create a list to hold results
    results = []

    print(f"Processing {len(df_questions)} questions...\n")

    for idx, row in df_questions.iterrows():
        user_question = row['Question']
        print(f"Processing Question {idx+1}: {user_question}")

        try:
            # Main generation call
            bot_response = retrieve_and_generate(user_question)
        except Exception as e:
            print(f"Error processing question {idx+1}: {e}")
            bot_response = "Error generating response."

        results.append({
            "Question": user_question,
            "Bot Response": bot_response
        })

    # Convert results to a DataFrame
    df_results = pd.DataFrame(results)

    # Save the results to CSV
    df_results.to_csv(output_csv_path, index=False)
    print(f"\nFinished! Responses saved to {output_csv_path}")

# Run the batch generation
batch_generate_responses("PPD_Questions.xlsx", "PPD_Generated_Responses.csv")


Processing 100 questions...

Processing Question 1: I feel disconnected from my baby, what should I do?


<ipython-input-25-62fa1601dce6>:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  max_similarity = float(max(similarities)) if similarities else 0.0
<ipython-input-22-913e3ca14515>:13: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  top_candidates = [candidates[int(i)] for i in sorted_indices[:top_k]]


Max similarity to retrieved docs: 0.87
Predicted Category: Parenting & Baby Care Stress
Processing Question 2: Am I not a good mother?


<ipython-input-25-62fa1601dce6>:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  max_similarity = float(max(similarities)) if similarities else 0.0
<ipython-input-22-913e3ca14515>:13: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  top_candidates = [candidates[int(i)] for i in sorted_indices[:top_k]]


Max similarity to retrieved docs: 0.83
Predicted Category: Relationship & Social Support
Processing Question 3: I need advice on how to deal with my spouse, I think he does not understand what I went through with childbirth?


...

Max similarity to retrieved docs: 0.65
Predicted Category: General Mental Health Support
Error processing question 98: Error code: 400 - {'error': {'message': "This model's maximum context length is 16385 tokens. However, you requested 16472 tokens (15972 in the messages, 500 in the completion). Please reduce the length of the messages or completion.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}
Processing Question 99: I cannot stop crying, I dont feel the love which a mother should feel, what is wrong with me?


<ipython-input-25-62fa1601dce6>:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  max_similarity = float(max(similarities)) if similarities else 0.0
<ipython-input-22-913e3ca14515>:13: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  top_candidates = [candidates[int(i)] for i in sorted_indices[:top_k]]


Max similarity to retrieved docs: 0.61
Predicted Category: Relationship & Social Support
Error processing question 99: Error code: 400 - {'error': {'message': "This model's maximum context length is 16385 tokens. However, you requested 16556 tokens (16056 in the messages, 500 in the completion). Please reduce the length of the messages or completion.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}
Processing Question 100: I dont want anyone to come visit me or the baby but people do not understand. I need time. I cry. What do I do?


<ipython-input-25-62fa1601dce6>:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  max_similarity = float(max(similarities)) if similarities else 0.0
<ipython-input-22-913e3ca14515>:13: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  top_candidates = [candidates[int(i)] for i in sorted_indices[:top_k]]


Max similarity to retrieved docs: 0.56
Predicted Category: General Mental Health Support
Error processing question 100: Error code: 400 - {'error': {'message': "This model's maximum context length is 16385 tokens. However, you requested 16464 tokens (15964 in the messages, 500 in the completion). Please reduce the length of the messages or completion.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}

Finished! Responses saved to PPD_Generated_Responses.csv


In [13]:
# Install Gradio if not already installed
!pip install gradio --quiet


In [14]:
import gradio as gr

# Make sure global variables are initialized BEFORE Gradio
conversation_history = []  # global conversation history


In [15]:
def gradio_chatbot(user_message, chat_history):
    global conversation_history

    if not user_message.strip():
        return "", chat_history

    try:
        bot_response = retrieve_and_generate(user_message)
    except Exception as e:
        print(f"Error inside chatbot: {e}")
        bot_response = "I'm sorry, I encountered an error. Please try again."

    conversation_history.append({"user": user_message, "bot": bot_response})

    chat_history.append(
        (f"🧑‍🍼 **You:** {user_message}", f"👩‍⚕️ **MOMCare:** {bot_response}")
    )

    return "", chat_history

def reset_conversation():
    global conversation_history
    conversation_history = []
    return []


In [16]:
with gr.Blocks() as demo:
    gr.Markdown("""
    # 👩‍⚕️ MOMCare: Postpartum Depression Support
    _Compassionate support for new mothers._ 🤱

    Feel free to share how you're feeling, or ask any questions. 💬
    """)

    chatbot = gr.Chatbot(
        label="MOMCare Support Chat",
        height=350,     # SMALLER window
        bubble_full_width=False
    )

    with gr.Row():
        with gr.Column(scale=8):
            user_input = gr.Textbox(
                placeholder="Type your message here...",
                show_label=False,
                lines=1,
                max_lines=3,
            )
        with gr.Column(scale=2):
            submit_btn = gr.Button("Send", variant="primary")
            clear_btn = gr.Button("Clear Chat", variant="secondary")

    submit_btn.click(gradio_chatbot, [user_input, chatbot], [user_input, chatbot])
    clear_btn.click(reset_conversation, outputs=[chatbot])

demo.launch(share=True)


<ipython-input-16-1607140901>:9: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
<ipython-input-16-1607140901>:9: DeprecationWarning: The 'bubble_full_width' parameter is deprecated and will be removed in a future version. This parameter no longer has any effect.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aace0a8b74155691c1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
from IPython.display import Javascript
Javascript('''
function ClickConnect(){
    console.log("Clicking");
    document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)
''')


<IPython.core.display.Javascript object>

In [ ]:
import time

while True:
    print("Running to keep the session alive...")
    time.sleep(60)  # Wait for 60 seconds

Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
Running to keep the session alive...
R